## **Resources**

[CodingForAll_Channels](https://www.youtube.com/watch?v=D43IitXdqk0)

[Official_Channel_Docs](https://channels.readthedocs.io/en/latest/introduction.html)

<hr>


## **ASGI**

[Official_Docs_ASGI](https://asgi.readthedocs.io/en/latest/introduction.html)

`ASGI` (Asynchronous Server Gateway Interface) is a specification that defines how web servers communicate with web applications in an asynchronous manner. It is designed to handle long-lived connections, such as those used in `WebSocket` communication, and allows for concurrent handling of multiple requests.

Where `WSGI` provided a standard for `synchronous` Python apps, `ASGI` provides one for both `asynchronous` and `synchronous` apps, with a `WSGI` `backwards-compatibility` implementation and multiple servers and application frameworks.

**What's Wrong with WSGI?**

You may ask “why not just upgrade WSGI”? This has been asked many times over the years, and the problem usually ends up being that `WSGI’s single-callable` interface just isn’t suitable for more involved Web protocols like `WebSocket`.

`WSGI` applications are a single, `synchronous callable` that takes a `request` and returns a `response`; this doesn’t allow for `long-lived connections`, like you get with long-poll `HTTP` or `WebSocket connections`.

<hr>

### **How Does `ASGI` Work?**

ASGI is structured as a single, `asynchronous callable`. It takes a `scope`, which is a `dict` containing details about the specific connection, `send`, an `asynchronous callable`, that lets the application send event messages to the client, and `receive`, an asynchronous callable which lets the application receive event messages from the client.

This not only allows multiple incoming events and outgoing events for each application, but also allows for a background `coroutine` so the application can do other things (such as listening for events on an external trigger, like a `Redis queue`).

In its simplest form, an application can be written as an asynchronous function, like this:

```python
async def application(scope, receive, send):
    event = await receive()
    ...
    await send({"type": "websocket.send", ...})
```

Every `event` that you send or receive is a Python `dict`, with a predefined format. It’s these event formats that form the basis of the standard, and allow applications to be swappable between servers.

These events each have a defined `type` key, which can be used to infer the event’s structure. Here’s an example event that you might receive from `receive` with the body from a HTTP request:

**HTTP Request Event**

```python
{
    "type": "http.request",
    "body": b"Hello, world!",
    "more_body": False,
}
```

And here’s an example of an event you might pass to `send` to send an outgoing WebSocket message:

**WebSocket Receive Event**

```python
{
    "type": "websocket.receive",
    "text": "Hello, world!",
}
```

`ASGI` is also designed to be a superset of `WSGI`, and there’s a defined way of translating between the two, allowing WSGI applications to be run inside ASGI servers through a translation wrapper (provided in the `asgiref` library). A `threadpool` can be used to run the synchronous WSGI applications away from the `async event loop`.

### **Components of ASGI**

**Protocol Server** : A `protocol server`, which terminates `sockets` and translates them into connections and per-connection event messages.

**Application** : An `application`, which lives inside a protocol server, is called once per connection, and handles event messages as they happen, emitting its own event messages back when necessary.

Like `WSGI`, the server hosts the application inside it, and dispatches incoming requests to it in a standardized format.

Unlike WSGI, however, applications are `asynchronous callables` rather than simple callables, and they communicate with the server by receiving and sending asynchronous `event messages` rather than receiving a single input stream and returning a single iterable.

`ASGI` applications must run as `async` / `await` compatible `coroutines` (i.e. `asyncio`-compatible) (on the main thread; they are free to use threading or other processes if they need synchronous code).

Unlike WSGI, there are two separate parts to an ASGI connection:

- `Connection scope` : A `scope` is a `dict` containing details about the specific connection, such as the protocol type, path, headers, and other metadata.

- `Event messages` : `Event messages` are `dicts` that represent incoming and outgoing events for the connection. They are sent and received using the `send` and `receive` callables provided to the application.

### **ASGI Middleware**

It is possible to have `ASGI “middleware”` - code that plays the role of both server and application, taking in a scope and the send/receive awaitable callables, potentially modifying them, and then calling an inner application.

When middleware is modifying the `scope`, it should make a copy of the `scope` object before mutating it and passing it to the inner application, as changes may leak upstream otherwise.

In particular, you should not assume that the copy of the `scope` you pass down to the application is the one that it ends up using, as there may be other middleware in the way; thus, do not keep a reference to it and try to mutate it outside of the initial ASGI app call.

Your one and only chance to add to it is before you hand control to the child application.

<hr>


<hr>
<hr>
<hr>


`HTTP` are stateless protocols. Each request from a client to a server is treated as an independent transaction that is unrelated to any previous request. This means that the server does not retain any information about the client's state between requests.

But, `WebSocket` is a stateful protocol. Once a `WebSocket` connection is established between a client and a server, it remains open and allows for continuous communication. Both the client and server can send messages to each other at any time without the need to re-establish the connection for each message.

## **Intro to `Django Channels`**

`Django Channels` is an extension to the Django web framework that enables handling of asynchronous protocols like `WebSocket`, `HTTP2`, and others. It allows Django applications to support real-time features such as chat applications, live notifications, and more.

`Channel` works on top of `ASGI` (Asynchronous Server Gateway Interface), which is the asynchronous counterpart to `WSGI` (Web Server Gateway Interface) used by traditional Django applications. This allows Django to handle multiple connections simultaneously and efficiently.

**Channels**

Channels are a way to handle multiple types of connections in Django. They allow you to define different types of consumers that can handle different protocols, such as `HTTP` and `WebSocket`. Channels use a layer called `Channel Layer` to facilitate communication between different parts of the application.

**Channel Layer**

The `Channel Layer` is a communication system that allows different parts of a Django application to send and receive messages. It acts as a bridge between different consumers and enables them to communicate with each other. The `Channel Layer` can be backed by various messaging systems, such as `Redis` or `RabbitMQ`.

<hr>

### **Scopes and Events**

`Channels` and `ASGI` split up incoming connections into two components: a `scope`, and a series of `events`.

**Scope**

The `scope` is a set of details about a single incoming connection - such as the path a web request was made from, or the originating IP address of a WebSocket, or the user messaging a chatbot. The `scope` persists throughout the connection.

For `HTTP`, the `scope` just lasts a single request.

For `WebSockets`, it lasts for the `lifetime` of the socket (but changes if the socket closes and reconnects).

For other `protocols`, it varies based on how the protocol’s ASGI spec is written; for example, it’s likely that a chatbot protocol would keep one scope open for the entirety of a user’s conversation with the bot, even if the underlying chat protocol is stateless.

**Events**

During the lifetime of this `scope`, a series of `events` occur.

These represent user interactions - `making a HTTP request`, for example, or `sending a WebSocket frame`.

Your `Channels` or `ASGI` applications will be instantiated `once per scope`, and then be fed the `stream of events` happening within that `scope` to decide what action to take.

An example with HTTP:

1. A user makes a `GET` request to `/home/`.

2. We open up a new `http` type scope with details of the request’s path, method, headers, etc.

3. We send a `http.request` event to our application with the body of the request.

4. The `Channels` or `ASGI` application processes this and generates a `http.response` event to send back to the browser and close the connection.

5. The HTTP request/response is completed and the scope is destroyed.

An example with a chatbot:

1. The user sends a first message to the chatbot.

2. This opens a `scope` containing the user’s username, chosen name, and user ID.

3. The application is given a `chat.received_message` event with the event text. It does not have to respond, but could send one, two or more other chat messages back as `chat.send_message` events if it wanted to.

4. The user sends more messages to the chatbot and more `chat.received_message` events are generated.

5. After a timeout or when the application process is restarted the scope is closed.

Within the lifetime of a scope - be that a chat, an HTTP request, a socket connection or something else - you will have one application instance handling all the events from it, and you can persist things onto the application instance as well.

You can choose to write a raw ASGI application if you wish, but `Channels` gives you an easy-to-use abstraction over them called **`Consumers`**.

<hr>


## **What is a Consumer?**

[Official_Docs_Consumers](https://channels.readthedocs.io/en/latest/topics/consumers.html) Read this Doc in detail to understand `Consumers` better.

A `consumer` is the basic unit of Channels code. We call it a `consumer` as it `consumes events`, but you can think of it as its own tiny little application.

When a request or new socket comes in, Channels will follow its routing table - we’ll look at that in a bit - find the right consumer for that incoming connection, and start up a copy of it.

This means that, unlike Django `views`, `consumers` are `long-running`. They can also be `short-running` - after all, `HTTP` requests can also be served by consumers - but they’re built around the idea of living for a little while (they live for the duration of a scope, as we described above).

A basic consumer looks like this:

```python

class ChatConsumer(WebsocketConsumer):

    def connect(self):
        self.username = "Anonymous"
        self.accept()
        self.send(text_data="[Welcome %s!]" % self.username)

    def receive(self, *, text_data):
        if text_data.startswith("/name"):
            self.username = text_data[5:].strip()
            self.send(text_data="[set your username to %s]" % self.username)
        else:
            self.send(text_data=self.username + ": " + text_data)

    def disconnect(self, message):
        pass
```

Each different protocol has different kinds of events that happen, and each type is represented by a different method. You write code that handles each event, and Channels will take care of scheduling them and running them all in parallel.

In the above example, we have a `WebSocket` consumer that handles three events:

- `connect`: called when the socket is opened.

- `receive`: called when a message is received on the socket.

- `disconnect`: called when the socket is closed.

The consumer keeps track of the user’s name as state on the instance, and updates it when the user sends a `/name` command.

When a new WebSocket connection is made, Channels will create a new instance of `ChatConsumer`, call its `connect` method, and then start sending it `receive` events as messages come in. When the socket closes, it will call the `disconnect` method and then destroy the instance.

Underneath, `Channels` is running on a fully `asynchronous event loop`, and if you write code like above, it will get called in a synchronous thread. This means you can safely do blocking operations, like calling the Django ORM:

```python
class LogConsumer(WebsocketConsumer):

    def connect(self, message):
        Log.objects.create(
            type="connected",
            client=self.scope["client"],
        )
```

However, if you want more control and you’re willing to work only in asynchronous functions, you can write fully asynchronous consumers:

```python
class PingConsumer(AsyncConsumer):
    async def websocket_connect(self, message):
        await self.send({
            "type": "websocket.accept",
        })

    async def websocket_receive(self, message):
        await asyncio.sleep(1)
        await self.send({
            "type": "websocket.send",
            "text": "pong",
        })
```

<hr>

### **Detailed About Consumers**

`Consumers` are just Python classes that define a set of methods that correspond to different types of `events` that can occur within a given `protocol`. Each method is called when the corresponding event occurs, and the consumer can then take appropriate action based on the event.

A `Consumer` is a subclass of either `SyncConsumer` or `AsyncConsumer`, depending on whether you want to write `synchronous` or `asynchronous` code.

**`SyncConsumer`**

Let’s look at a basic example of a `SyncConsumer`:

```python
from channels.consumer import SyncConsumer

class EchoConsumer(SyncConsumer):

    def websocket_connect(self, event):
        self.send({
            "type": "websocket.accept",
        })

    def websocket_receive(self, event):
        self.send({
            "type": "websocket.send",
            "text": event["text"],
        })
```

This is a very simple WebSocket echo server - it will accept all incoming WebSocket connections, and then reply to all incoming WebSocket text frames with the same text.

Consumers are structured around a series of named methods corresponding to the `type` value of the messages they are going to receive, with any `.` replaced by `_`. The two handlers above are handling `websocket.connect` and `websocket.receive` messages respectively.

But,

How did we know what event types we were going to get and what would be in them (like `websocket.receive` having a `text`) key?

That’s because we designed this against the ASGI WebSocket specification, which tells us how WebSockets are presented - read more about it in ASGI - and protected this application with a router that checks for a scope type of `websocket` - see more about that in [Routing](https://channels.readthedocs.io/en/latest/topics/routing.html).

**`AsyncConsumer`**

The `AsyncConsumer` is laid out very similarly, but all the handler methods must be `coroutines`, and `self.send` is a coroutine:

```python

from channels.consumer import AsyncConsumer

class EchoConsumer(AsyncConsumer):

    async def websocket_connect(self, event):
        await self.send({
            "type": "websocket.accept",
        })

    async def websocket_receive(self, event):
        await self.send({
            "type": "websocket.send",
            "text": event["text"],
        })

```

**When to Use `SyncConsumer` vs `AsyncConsumer`?**

The main thing to consider is what you’re talking to. If you call a slow `synchronous` function from inside an `AsyncConsumer` you’re going to hold up the entire `event loop`, so they’re only useful if you’re also calling `async code` (for example, using `HTTPX` to fetch 20 pages in parallel).

If you’re calling any part of `Django’s ORM` or other `synchronous` code, you should use a `SyncConsumer`, as this will run the whole `consumer` in a `thread` and stop your ORM queries blocking the entire server.

It is recommended to write `SyncConsumers` by default, and only use `AsyncConsumers` in cases where you know you are doing something that would be improved by async handling (long-running tasks that could be done in parallel) and you are only using `async-native` libraries.

If we need to call synchronous code from an `AsyncConsumer`, we can use `database_sync_to_async` or `sync_to_async` decorators to run the synchronous code in a separate thread, preventing it from blocking the event loop.

```python
from channels.db import database_sync_to_async
class MyAsyncConsumer(AsyncConsumer):

    @database_sync_to_async
    def get_user(self, user_id):
        return User.objects.get(id=user_id)
    async def websocket_connect(self, event):
        user = await self.get_user(self.scope["user"].id)
        # Do something with the user
```

This way, we can safely call synchronous code without blocking the event loop in our `AsyncConsumer`.

<hr>

### **Closing Consumers**

When the socket or connection attached to your consumer is `closed` - either by you or the client - you will likely get an event sent to you (for example, `http.disconnect` or `websocket.disconnect`), and your application instance will be given a short amount of time to act on it.

If you need to do any cleanup - for example, removing the user from a chat room, or closing database connections - you should do it in the appropriate `disconnect handler`.

Once you have finished doing your `post-disconnect cleanup`, you need to raise `channels.exceptions.StopConsumer` to halt the `ASGI` application `cleanly` and let the server clean it up.

If you leave it running - by not raising this exception - the server will reach its application close timeout (which is 10 seconds by default in `Daphne`) and then kill your application and raise a `warning`.

But,

The generic `Consumers` like `WebSocketConsumer` and `AsyncWebSocketConsumer` already handle this for you, so this is only needed if you are writing your own consumer class based on `AsyncConsumer` or `SyncConsumer`.

However, if you override their `__call__` method, or block the handling methods that it calls from returning, you may still run into this; take a look at their source code if you want more information.

Additionally, if you launch your own background coroutines, make sure to also shut them down when the connection is finished, or you’ll leak coroutines into the server.

<hr>

### **`Consumer` with `Channel Layers`**

As we know, `Channel Layers` allow different parts of a Django application to communicate with each other by sending and receiving messages.

So, `Consumers` can use `Channel Layers` to let them send messages between each `other either` one-to-one or via a `broadcast` system called **`groups`**.

`Consumers` will use the channel layer `default` unless the `channel_layer_alias` attribute is set when subclassing any of the provided `Consumer` classes.

To use the channel layer echo_alias we would set it like so:

```python
from channels.consumer import SyncConsumer

class EchoConsumer(SyncConsumer):
    channel_layer_alias = "echo_alias"
```

You can read more in [Channel_Layers](https://channels.readthedocs.io/en/latest/topics/channel_layers.html).

<hr>

### **Scope**

`Consumers` receive the connection’s `scope` when they are called, which contains a lot of the information you’d find on the `request` object in a Django view. It’s available as `self.scope` inside the consumer’s methods.

Scopes are part of the [ASGI_specification], but here are some common things you might want to use:

- `self.scope["type"]`: The type of connection - for example, `http` or `websocket`.

- `self.scope["path"]`: The path the connection was made to - for example, `/chat/room/1/`.

- `self.scope["query_string"]`: The query string of the connection, as a bytestring.

- `self.scope["headers"]`: A list of `(name, value)` tuples representing the headers sent with the connection. We can access `Authorization` header with this.

- `self.scope["method"]`: The HTTP method of the request (only for `http` type scopes).

If you enable things like `Authentication`, you’ll also be able to access the user object as `scope["user"]`, and the `URLRouter`, for example, will put captured groups from the URL into `scope["url_route"]`.

In general, the scope is the place to get connection information and where middleware will put attributes it wants to let you access (in the same way that `Django’s middleware` adds things to `request`).

<hr>

### **Generic Consumers**

What you see above is the basic layout of a consumer that works for any protocol. Much like Django’s `generic views`, Channels ships with `generic consumers` that wrap common functionality up so you don’t need to rewrite it, specifically for `HTTP` and `WebSocket` handling.

**`WebSocketConsumer`**

Available as `channels.generic.websocket.WebsocketConsumer`, this wraps the verbose plain-ASGI message sending and receiving into handling that just deals with `text` and `binary frames`:

```python
from channels.generic.websocket import WebsocketConsumer

class MyConsumer(WebsocketConsumer):
    groups = ["broadcast"]

    def connect(self):
        # Called on connection.
        # To accept the connection call:
        self.accept()
        # Or accept the connection and specify a chosen subprotocol.
        # A list of subprotocols specified by the connecting client
        # will be available in self.scope['subprotocols']
        self.accept("subprotocol")
        # To reject the connection, call:
        self.close()

    def receive(self, text_data=None, bytes_data=None):
        # Called with either text_data or bytes_data for each frame
        # You can call:
        self.send(text_data="Hello world!")
        # Or, to send a binary frame:
        self.send(bytes_data="Hello world!")
        # Want to force-close the connection? Call:
        self.close()
        # Or add a custom WebSocket error code!
        self.close(code=4123)

    def disconnect(self, close_code):
        # Called when the socket closes
```

You can also raise `channels.exceptions.AcceptConnection` or `channels.exceptions.DenyConnection` from anywhere inside the `connect` method in order to accept or reject a connection, if you want reusable authentication or rate-limiting code that doesn’t need to use mixins.

A `WebsocketConsumer’s` channel will automatically be added to (on connect) and removed from (on disconnect) any groups whose names appear in the consumer’s `groups` class attribute. `groups` must be an iterable, and a channel layer with support for groups must be set as the channel backend (`channels.layers.InMemoryChannelLayer` and `channels_redis.core.RedisChannelLayer` both support groups). If no channel layer is configured or the channel layer doesn’t support groups, connecting to a `WebsocketConsumer` with a non-empty groups attribute will raise `channels.exceptions.InvalidChannelLayerError`. See [Groups](https://channels.readthedocs.io/en/latest/topics/channel_layers.html#groups) for more.

**`AsyncWebSocketConsumer`**

Available as `channels.generic.websocket.AsyncWebsocketConsumer`, this has the exact same methods and signature as `WebsocketConsumer` but everything is `async`, and the functions you need to write have to be as well:

```python

from channels.generic.websocket import AsyncWebsocketConsumer

class MyConsumer(AsyncWebsocketConsumer):
    groups = ["broadcast"]

    async def connect(self):
        # Called on connection.
        # To accept the connection call:
        await self.accept()
        # Or accept the connection and specify a chosen subprotocol.
        # A list of subprotocols specified by the connecting client
        # will be available in self.scope['subprotocols']
        await self.accept("subprotocol")
        # To reject the connection, call:
        await self.close()

    async def receive(self, text_data=None, bytes_data=None):
        # Called with either text_data or bytes_data for each frame
        # You can call:
        await self.send(text_data="Hello world!")
        # Or, to send a binary frame:
        await self.send(bytes_data="Hello world!")
        # Want to force-close the connection? Call:
        await self.close()
        # Or add a custom WebSocket error code!
        await self.close(code=4123)

    async def disconnect(self, close_code):
        # Called when the socket closes
```

**`JsonWebSocketConsumer`**

Available as `channels.generic.websocket.JsonWebsocketConsumer`, this works like `WebsocketConsumer`, except it will `auto-encode` and decode to `JSON` sent as `WebSocket` text frames.

The only API differences are:

- Your `receive_json` method must take a single argument, `content`, that is the decoded JSON object.

- `self.send_json` takes only a single argument, `content`, which will be encoded to JSON for you.

If you want to customise the JSON encoding and decoding, you can override the `encode_json` and `decode_json` classmethods.

<hr>


<hr>
<hr>
<hr>


## **Routing**

[Routing_Docs](https://channels.readthedocs.io/en/latest/topics/routing.html) Read this Doc in detail to understand `Routing` better.

We need to understand that each `Consumer` say `ChatConsumer` or `NotificationConsumer` is a self contained ASGI application that can be called with a `scope`, `receive` and `send` callables.

But in real world applications, we've multiple consumers that we want to route to based on the incoming connection's details, such as the URL path or the type of connection (HTTP, WebSocket, etc.). This is where `routing` comes into play. In such case, we can't have a single consumer handling all types of connections.

To handle this, `Channels` provides a routing `Classes` that allow us to combine and stack multiple consumers together and route incoming connections to the appropriate consumer based on the connection's details.

We can use `ProtocolTypeRouter`, `URLRouter`, and `PathRouter` which are `ASGI` applications that can be used to route incoming connections to the appropriate consumer based on the connection's details.

These `Classes` takes `scope` and check the `type` key to determine which protocol is being used (e.g., `http`, `websocket`, etc.) and dispatches the connection to the appropriate consumer based on that.

With this,

We can route `Http` requests to `Django views` and `WebSocket` connections to `WebSocket consumers`, or any other protocol, all within the same application.

**For Example:**

- If the connection is `type="http"`, dispatch to Django’s normal ASGI app.

- If the connection is `type="websocket"` and the path starts with `/ws/chat/`, dispatch to `ChatConsumer`.

- If it’s `/ws/notifications/`, dispatch to `NotificationConsumer`.

Therefore, we should have a `ProtocolTypeRouter` as the root application of your project - the one that you pass to protocol servers - and nest other, more protocol-specific routing underneath there.

**Note:** Channels routers only work on `scope` level , not on the `event` level. This means that once a consumer is chosen for a given scope, all events for that scope will go to that consumer.

**Example of Routing:**

```python

import os

from channels.auth import AuthMiddlewareStack
from channels.routing import ProtocolTypeRouter, URLRouter
from channels.security.websocket import AllowedHostsOriginValidator
from django.core.asgi import get_asgi_application
from django.urls import path

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "mysite.settings")
# Initialize Django ASGI application early to ensure the AppRegistry
# is populated before importing code that may import ORM models.
django_asgi_app = get_asgi_application()

from chat.consumers import AdminChatConsumer, PublicChatConsumer

application = ProtocolTypeRouter({
    # Django's ASGI application to handle traditional HTTP requests
    "http": django_asgi_app,

    # WebSocket chat handler
    "websocket": AllowedHostsOriginValidator(
        AuthMiddlewareStack(
            URLRouter([
                path("chat/admin/", AdminChatConsumer.as_asgi()),
                path("chat/", PublicChatConsumer.as_asgi()),
            ])
        )
    ),
})
```

We can also add `middleware` to the routing, such as `AuthMiddlewareStack` to handle authentication and `AllowedHostsOriginValidator` to validate the origin of incoming WebSocket connections.

**Note:**

- We call the `as_asgi()` classmethod when routing our consumers. This returns an ASGI wrapper application that will instantiate a new `consumer instance` for each connection or scope. This is similar to Django’s `as_view()`, which plays the same role for `per-request instances` of class-based views.

<hr>

### **Routers Available in Channels**

**`ProtocolTypeRouter`**

`channels.routing.ProtocolTypeRouter`

- This router routes incoming connections based on the `protocol type` (e.g., `http`, `websocket`, etc.). It allows you to define different consumers for different protocol types.

- This should be the top level of your ASGI application stack and the main entry in your routing file.

- It lets you dispatch to one of a number of other ASGI applications based on the type value present in the `scope`. Protocols will define a fixed type value that their `scope` contains, so you can use this to distinguish between incoming connection types.

- It takes a single argument - a dictionary mapping type names to ASGI applications that serve them:

```python
ProtocolTypeRouter({
    "http": some_app,
    "websocket": some_other_app,
})
```

**`URLRouter`**

`channels.routing.URLRouter`

- This router routes incoming connections based on the `URL path`. It allows you to define different consumers for different URL patterns.

- Routes `http` or `websocket` type connections via their HTTP path. Takes a single argument, a list of Django URL objects (either `path()` or `re_path()`):

```python
URLRouter([
    re_path(r"^longpoll/$", LongPollConsumer.as_asgi()),
    re_path(r"^notifications/(?P<stream>\w+)/$", LongPollConsumer.as_asgi()),
    re_path(r"", get_asgi_application()),
])
```

Any captured groups will be provided in `scope` as the key `url_route`, a dict with a `kwargs` key containing a dict of the named regex groups and an `args` key with a list of positional regex groups. Note that named and unnamed groups cannot be mixed: Positional groups are discarded as soon as a single named group is matched.

For example, to pull out the named group stream in the example above, you would do this:

```python
stream = self.scope["url_route"]["kwargs"]["stream"]
```

<hr>


<hr>
<hr>
<hr>


## **`Database Access` with Django Channels**

If we're using `WSGI` or `Django views`, database access is `synchronous`, and we can use the Django ORM directly. Meaning, when we use `filter()`, `get()`, or any other ORM method, it is blocking `I/O` operation, and the server will wait for the database to respond before moving on to the next request.

So, if we use `Django ORM` directly in `asynchronous` code, it can block the entire event loop, which defeats the purpose of async code and can even cause connection leaks and performance issues.

Therefore, when using `Django ORM` in `asynchronous` code, we need to `wrap` our database calls in a way that it runs in a separate `Thread` or `Process`, so it doesn't block the `event loop`.

<hr>

To run the `Django ORM` in a separate thread, we can use the `database_sync_to_async` decorator provided by `Channels`. This decorator will run the decorated function in a separate `thread`, allowing the event loop to continue processing other requests while waiting for the database operation to complete.

But,

After `Django 4.1+` we can use `Django's` native `Async ORM` methods such as `aget()`, `afilter()`, `acreate()`, etc. to perform database operations asynchronously without blocking the event loop.

<hr>

**Notes:**

- If you’re using `SyncConsumer`, or anything based on it - like `JsonWebsocketConsumer` - you don’t need to do anything special, as all your code is already run in a synchronous mode and `Channels` will do the cleanup for you as part of the `SyncConsumer` code.

### **Database Connections in Async Consumers**

As we know, `Channel` uses an `event loop` to handle multiple connections concurrently.

Therefore, we need to understand **Event Loop**.

**Event Loop**

In `Django Channels`, the `event loop` is the core component that manages and schedules the execution of asynchronous tasks. It allows multiple tasks to run concurrently without blocking each other, enabling efficient handling of I/O-bound operations like database access, network requests, and more.

`Event Loop` continuously checks for tasks that are ready to run and executes them in a non-blocking manner. When a task encounters an I/O operation, it yields control back to the event loop, allowing other tasks to run while waiting for the I/O operation to complete.

`Event Loop` runs in a single thread, which means that all tasks share the same thread of execution. This is different from traditional multi-threading, where each task runs in its own thread.


## **Tomorrow Cont**

https://developer.mozilla.org/en-US/docs/Web/API/Media_Capture_and_Streams_API

https://github.com/Iftimie/LearningWebRTC/tree/main/with_django/signalserver

https://gist.github.com/Iftimie/242a9c4c515490761e14f9690b810f62#file-connection_setup_caller-html

https://channels.readthedocs.io/en/latest/introduction.html#

https://github.com/aiortc/aiortc

https://github.com/yinguobing/arcface
